In [1]:
import pickle
import random
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from rdkit.Chem import AllChem
from rdchiral.main import rdchiralRunText

from anytree import AnyNode, RenderTree, PreOrderIter
from tqdm import tqdm

/home/wang_m@WMGDS.WMG.WARWICK.AC.UK/anaconda3/envs/drug/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
fp_dim = 2048
buffer_capacity = 10000
# memo

In [3]:
class RolloutPolicyNet(nn.Module):
    def __init__(self, n_rules, fp_dim=2048, dim=512, dropout_rate=0.3):
        super(RolloutPolicyNet, self).__init__()
        self.fp_dim = fp_dim
        self.n_rules = n_rules
        self.dropout_rate = dropout_rate
        self.fc1 = nn.Linear(fp_dim,dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.dropout1 = nn.Dropout(dropout_rate)
        # self.fc2 = nn.Linear(dim,dim)
        # self.bn2 = nn.BatchNorm1d(dim)
        # self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(dim,n_rules)

    def forward(self,x, y=None, loss_fn =nn.CrossEntropyLoss()):
        x = self.dropout1(F.elu(self.bn1(self.fc1(x))))
        # x = self.dropout1(F.elu(self.fc1(x)))
        # x = self.dropout2(F.elu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        if y is not None :
            return loss_fn(x, y)
        else :
            return x
        return x

def load_model(state_path, template_rule_path,fp_dim=2048):
    template_rules = {}
    with open(template_rule_path, 'r') as f:
        for i, l in enumerate(f):
            rule= l.strip()
            template_rules[rule] = i
    idx2rule = {}
    for rule, idx in template_rules.items():
        idx2rule[idx] = rule
    rollout = RolloutPolicyNet(len(template_rules),fp_dim=fp_dim)
    
    state_dict = torch.load(state_path, map_location='cpu')
    # Remove the 'module.' prefix if present (i.e., if it was saved using DataParallel)
    new_state_dict = {}
    for key in state_dict.keys():
        new_key = key.replace("module.", "")  # Remove 'module.' prefix
        new_state_dict[new_key] = state_dict[key]
    rollout.load_state_dict(new_state_dict)
    return rollout, idx2rule, template_rules
    
class MLPModel(object):
    def __init__(self, state_path, template_path, device=-1, fp_dim=2048, realistic_filter=False):
        super(MLPModel, self).__init__()
        self.fp_dim = fp_dim
        self.net, self.idx2rules, self.rules2idx = load_model(state_path, template_path, fp_dim)
        self.net.eval()
        self.device = device
        if device >= 0:
            self.net.to(device)

        self.realistic_filter = realistic_filter
        
        self.reference_net, _, _ = load_model(state_path, template_path, fp_dim)
        self.reference_net.eval()
        if device >= 0:
            self.reference_net.to(device)

    def forward_topk(self, arr, topk=10):
        preds = self.net(arr)
        preds_reference = self.reference_net(arr).detach()

        if self.realistic_filter:
            preds_reference, idx_topk = torch.topk(preds_reference, k=topk)
            preds = preds.gather(1, idx_topk)
        else:
            preds, idx_topk = torch.topk(preds, k=topk)
            preds_reference = preds_reference.gather(1, idx_topk)

        return preds, preds_reference, idx_topk

In [4]:
def preprocess(X,fp_dim):
    # Compute fingerprint from mol to feature
    mol = Chem.MolFromSmiles(X)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=int(fp_dim),useChirality=True)
    onbits = list(fp.GetOnBits())
    arr = np.zeros(fp.GetNumBits())
    arr[onbits] = 1
    # arr = (arr - arr.mean())/(arr.std() + 0.000001)
    # arr = arr / fp_dim
    # X = fps_to_arr(X)
    return arr

def prepare_starting_molecules(filename):
    if filename[-3:] == 'csv':
        starting_mols = set(list(pd.read_csv(filename)['mol']))
    else:
        assert filename[-3:] == 'pkl'
        with open(filename, 'rb') as f:
            starting_mols = pickle.load(f)
    return starting_mols

In [5]:
class RetroEnv:
    def __init__(self, mode='training'):
        self.mode = mode
        if mode == 'training':
            self.routes = pickle.load(open('dataset/routes_train.pkl', 'rb'))
            self.target_mols = [Chem.CanonSmiles(self.route[0].split('>')[0]) for self.route in tqdm(self.routes[:200], mininterval=2, desc='Load target molecules')]
        elif mode == 'evaluation':
            self.routes = pickle.load(open('dataset/routes_possible_test_hard.pkl', 'rb'))
            self.target_mols = [Chem.CanonSmiles(self.route[0].split('>')[0]) for self.route in tqdm(self.routes, mininterval=2, desc='Load target molecules')]
        self.building_blocks = prepare_starting_molecules('dataset/origin_dict_canonical.csv')
    
    def reset(self, mol_idx=None):
        self.tree_mdp = None
        self.tobevisited_nodes = []

        if mol_idx == None:
            self.obs = random.choice(self.target_mols)
        else:
            self.obs = self.target_mols[mol_idx]
            
        self.obs_node = AnyNode(id=self.obs, action=None, inbb=int(self.obs in self.building_blocks))
        self.tree_mdp = self.obs_node
        return self.obs

    def step(self, a):
        self.obs_node.action = a
        reactants = rdchiralRunText(a, self.obs)
        if len(reactants) == 0:
            return None, 0, 1, None
        else:
            reactants = list(set(reactants[0].split('.')))
        
        for r in reactants:
            r_node = AnyNode(id=r, parent=self.obs_node, action=None, inbb=int(r in self.building_blocks))
            if not r in self.building_blocks:
                self.tobevisited_nodes.append(r_node)
        
        if self.tobevisited_nodes:
            self.obs = self.tobevisited_nodes[-1].id
            self.obs_node = self.tobevisited_nodes[-1]
            self.tobevisited_nodes.remove(self.tobevisited_nodes[-1])
            return self.obs, 0, 0, None
        else:
            self.obs = None
            self.obs_node = None
            return None, 1, 1, None

        #failed_leaves = [node for node in PreOrderIter(self.tree_mdp) if node.is_leaf and (node not in self.building_blocks)]
        
    


In [6]:
class RetroRL:
    def __init__(self, topk=10, fp_dim=2048, sampling=True):
        self.fp_dim = fp_dim
        self.topk = 10
        self.sampling = sampling
        self.one_step = MLPModel(state_path='one_step_model/saved_rollout_state_1_2048.ckpt', 
                                 template_path='one_step_model/template_rules_1.dat', device=0)
    def get_action(self, a): # a is a SMILES
        arr = preprocess(a, self.fp_dim)
        arr = np.reshape(arr,[-1, arr.shape[0]])
        arr = torch.tensor(arr, dtype=torch.float32)
        arr = arr.to(device='cuda')
        preds, preds_reference, idx = self.one_step.forward_topk(arr, topk=self.topk) 
        if self.sampling:
            probs = F.softmax(preds, dim=1)
            probs_reference = F.softmax(preds_reference, dim=1)
            action_class = torch.multinomial(probs, num_samples=1).squeeze().cpu().numpy()
            reference_class = torch.multinomial(probs_reference, num_samples=1).squeeze().cpu().numpy()
        else: 
            action_class = torch.argmax(preds, dim=1).cpu().numpy()
            reference_class = torch.argmax(preds_reference, dim=1).cpu().numpy()
        action_indices = idx[0][action_class].cpu().item()
        reference_indices = idx[0][reference_class].cpu().item()

        ac_reaction = self.one_step.idx2rules[action_indices]
        ref_reaction = self.one_step.idx2rules[reference_indices]
        return ac_reaction, ref_reaction

In [7]:
agent = RetroRL(sampling=False)
env = RetroEnv()
eval_env = RetroEnv(mode='evaluation')

Load target molecules: 100%|████████████████| 190/190 [00:00<00:00, 5459.87it/s]


In [75]:
def eval(envs, agent):
    eval_info = {}
    for env in envs:
        rewards = np.zeros((len(env.target_mols), 1))
        for mol_idx in range(min(len(env.target_mols), 200)):
            obs = env.reset(mol_idx)
            for t in range(100):
                a, _ = agent.get_action(obs)
                next_obs, reward, done, info = env.step(a)
                rewards[mol_idx] = reward
                if done:
                    break
                obs = next_obs
        eval_info[env.mode+'/success rate'] = rewards.mean()
    return eval_info

In [9]:
eval_info = eval([eval_env, env], agent)
print(eval_info)

Evaluate the model (training): 100%|██████████| 200/200 [00:06<00:00, 32.94it/s]

{'evaluation/success rate': 0.1631578947368421, 'training/success rate': 0.925}


In [10]:
def is_node_valid(node):
    # Check if the node is valid (all leaf nodes have inbb == 1)
    if not node.children:
        return node.inbb == 1
    return all(is_node_valid(child) for child in node.children)

def find_valid_subtrees(node, valid_subtrees):
    if node is None:
        return

    # Check if the current node is valid and has children
    if is_node_valid(node) and node.children:
        valid_subtrees.append(node)

    # Recursively check each child
    for child in node.children:
        find_valid_subtrees(child, valid_subtrees)

In [11]:
def is_ancestor(ancestor, node):
    current = node
    while current is not None:
        if current == ancestor:
            return True
        current = current.parent
    return False

def remove_joint_subtrees(root, valid_subtrees):
    # Create a set to track nodes to remove
    to_remove = set()
    
    # Check each node against all other nodes
    for node in valid_subtrees:
        for other_node in valid_subtrees:
            if node != other_node and is_ancestor(node, other_node):
                to_remove.add(other_node)
    
    # Remove nodes that are descendants of any node in the list
    filtered_nodes = [node for node in valid_subtrees if node not in to_remove]
    return filtered_nodes

In [12]:
def get_paths_from_tree(node, path=None, reward=0):
    if path is None:
        path = []  
    path.append(node.id)
    path.append(node.action)
    
    if not node.children:
        if reward == 1:
            return [list(path)]
        return [] 

    all_paths = []
    for child in node.children:
        all_paths += get_paths_from_tree(child, path.copy(), reward=child.inbb)
    return all_paths

In [32]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim, action_dim):
        """
        Initialize the replay buffer.

        :param capacity: Maximum number of experiences to store.
        :param obs_dim: Dimension of the observation space (e.g., 2048 for 2048-dimensional observations).
        :param action_dim: Dimension of the action space (discrete or continuous).
        """
        self.capacity = capacity
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.buffer = {
            'obs': np.zeros((capacity, obs_dim), dtype=np.float32),
            'next_obs': np.zeros((capacity, obs_dim), dtype=np.float32),
            'actions': np.zeros((capacity, action_dim), dtype=np.float32),
            'rewards': np.zeros(capacity, dtype=np.float32),
            'dones': np.zeros(capacity, dtype=np.float32),
        }
        self.size = 0
        self.pos = 0

    def add(self, obs, next_obs, action, reward, done):
        """
        Add an experience to the buffer.

        :param obs: Current observation (shape: [obs_dim]).
        :param next_obs: Next observation (shape: [obs_dim]).
        :param action: Action taken (shape: [action_dim]).
        :param reward: Reward received (float).
        :param done: Whether the episode has ended (bool).
        """
        idx = self.pos % self.capacity
        self.buffer['obs'][idx] = obs
        self.buffer['next_obs'][idx] = next_obs
        self.buffer['actions'][idx] = action
        self.buffer['rewards'][idx] = reward
        self.buffer['dones'][idx] = done
        self.pos += 1
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        """
        Sample a batch of experiences from the buffer.

        :param batch_size: Number of experiences to sample.
        :return: A dictionary of sampled experiences.
        """
        indices = random.sample(range(self.size), batch_size)
        return {
            'obs': self.buffer['obs'][indices],
            'next_obs': self.buffer['next_obs'][indices],
            'actions': self.buffer['actions'][indices],
            'rewards': self.buffer['rewards'][indices],
            'dones': self.buffer['dones'][indices],
        }

    def __len__(self):
        return self.size

buffer = ReplayBuffer(capacity=buffer_capacity, obs_dim=fp_dim, action_dim=len(agent.one_step.rules2idx))

In [33]:
def translate_trajs(trajs):
    for traj in trajs:
        for i in range(len(traj)-1):
            if i%2 == 0:
                traj[i] = preprocess(traj[i], fp_dim)
            else:
                rxn_idx = agent.one_step.rules2idx[traj[i]]
                traj[i] = np.zeros(len(agent.one_step.idx2rules))
                traj[i][rxn_idx] = 1
        traj.pop()
    return trajs

In [34]:
def collect_trajs(env, agent):
    obs = env.reset()
    for t in range(100):
        a, _ = agent.get_action(obs)
        next_obs, reward, done, info = env.step(a)
        if done:
            break
        obs = next_obs

    valid_subtrees = []
    find_valid_subtrees(env.tree_mdp, valid_subtrees)
    valid_subtrees = remove_joint_subtrees(env.tree_mdp, valid_subtrees)

    traj = []
    for root in valid_subtrees:
        traj += get_paths_from_tree(root)
    return traj

In [35]:
trajs = collect_trajs(env, agent)

In [36]:
for pre, fill, node in RenderTree(env.tree_mdp):
    print("%s%s----%s" % (pre, node.id, node.inbb))

CCNC(=O)c1nn(-c2ccc(Cl)cc2)c(=O)c2c(N)scc12----0
└── CC(C)NC(=O)c1nn(-c2ccc(Cl)cc2)c(=O)c2c(N)scc12----0
    ├── CCN(C(C)C)C(C)C----1
    └── Nc1scc2c(C(=O)O)nn(-c3ccc(Cl)cc3)c(=O)c12----0
        └── CCOC(=O)c1nn(-c2ccc(Cl)cc2)c(=O)c2c(N)scc12----1


In [17]:
def translate_trajs(trajs):
    for traj in trajs:
        for i in range(len(traj)-1):
            if i%2 == 0:
                traj[i] = preprocess(traj[i], fp_dim)
            else:
                rxn_idx = agent.one_step.rules2idx[traj[i]]
                traj[i] = np.zeros(len(agent.one_step.idx2rules))
                traj[i][rxn_idx] = 1
        traj.pop()
    return trajs

In [45]:
import torch
import torch.nn as nn

In [53]:
class ValueMLP(nn.Module):
    def __init__(self, n_layers, fp_dim, latent_dim, dropout_rate, device):
        super(ValueMLP, self).__init__()
        self.n_layers = n_layers
        self.fp_dim = fp_dim
        self.latent_dim = latent_dim
        self.dropout_rate = dropout_rate
        self.device = device

        layers = []
        layers.append(nn.Linear(fp_dim, latent_dim))
        # layers.append(nn.BatchNorm1d(latent_dim,
        #                              track_running_stats=False))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(self.dropout_rate))
        for _ in range(self.n_layers - 1):
            layers.append(nn.Linear(latent_dim, latent_dim))
            # layers.append(nn.BatchNorm1d(latent_dim,
            #                              track_running_stats=False))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(self.dropout_rate))
        layers.append(nn.Linear(latent_dim, 1))

        self.layers = nn.Sequential(*layers)

    def forward(self, fps):
        x = fps
        x = self.layers(x)
        x = torch.log(1 + torch.exp(x))

        return x
        
value_net = ValueMLP(n_layers=1, fp_dim=fp_dim, latent_dim=128, dropout_rate=0.1, device='cuda').to(device='cuda')
value_opt = torch.optim.Adam(value_net.parameters(), lr=0.0001)
value_net.eval()
policy_opt = torch.optim.Adam(agent.one_step.net.parameters(), lr=0.0001)
_ = agent.one_step.net.eval()

In [ ]:
for episode in tqdm(range(5000)):
    trajs = collect_trajs(env, agent)
    translated_trajs = translate_trajs(trajs)
    for tj in translated_trajs:
        for i in range(0, len(tj)-1, 2):
            buffer.add(obs=tj[i], next_obs=tj[i+2], action=tj[i+1], reward=1, done=0)

    for _ in range(20):
        batch = buffer.sample(128)
        obs = torch.Tensor(batch['obs']).to(device='cuda')
        acs = torch.Tensor(batch['actions']).to(device='cuda')
        preds = agent.one_step.net(obs)
        
        bce_loss = nn.BCEWithLogitsLoss()
        loss = bce_loss(preds, acs)
        policy_opt.zero_grad()
        loss.backward()
        policy_opt.step()

    if episode % 100 == 0:
        eval_info = eval([eval_env, env], agent)
        print(str(episode), eval_info)


  0%|                                       | 1/5000 [01:10<98:11:18, 70.71s/it]

0 {'evaluation/success rate': 0.15263157894736842, 'training/success rate': 0.93}


  2%|▋                                    | 101/5000 [04:04<21:26:38, 15.76s/it]

100 {'evaluation/success rate': 0.17894736842105263, 'training/success rate': 0.945}


  4%|█▍                                   | 201/5000 [06:35<12:10:58,  9.14s/it]

200 {'evaluation/success rate': 0.18947368421052632, 'training/success rate': 0.945}


  6%|██▎                                   | 301/5000 [09:01<8:53:39,  6.81s/it]

300 {'evaluation/success rate': 0.20526315789473684, 'training/success rate': 0.945}


  7%|██▍                                   | 328/5000 [09:35<1:40:15,  1.29s/it]

In [62]:
batch = buffer.sample(128)
obs = torch.Tensor(batch['obs'])

In [68]:
preds = agent.one_step.net(obs)

In [71]:
acs

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')